# NB15b — Multi-Seed Interaction Robustness (Phase D.1 Followup)

**Lock-Basis:** [ANN-016 FX as Reference Blueprint](../docs/decisions/ANN-016-fx-as-reference-blueprint-industrialization-first.md) — D.1 Continuation nach NB15a.

## Mission

NB15a hat zwei Befunde geliefert, die zusammen die Forschungsfrage neu definieren:

1. **USDCHF ist single-seed NICHT broken** (seed=7, PF 1.56) — vs Multi-Seed-Aggregat NB14f-v2 PF 0.17. Seed-fragil, nicht strukturell invertiert.
2. **Filter-Interaction ist destruktiv NUR bei USDCHF** — Base+HTF+Session zerstört sich (1.55 → 1.67 → 1.70 → 1.56). Andere 3 Pairs sind additiv.

**Neue Forschungsfrage (Nico-Direktive):**
> "Wie robust ist das FX-System gegenüber Seed- und Filter-Interaction-Instabilität?"

**KEINE ANN-017 vor diesem Notebook.** Wir brauchen erst eine stabile Problemdefinition.

## Setup (gleiche Pipeline, NUR seeds variieren)

- Gleiche Daten (Pool-Expansion: 3 Train, 4 Hold-Out)
- Gleiche Filter (HTF + NY-Session)
- Identische evaluation pipeline (`filter_marginal_contributions` aus `core.analysis.diagnostic_decomposer`)
- **Seeds:** 42, 1, 7 (matched mit NB14f-v2 Multi-Seed-Set)

## Output (Nico's Wunsch-Tabelle)

```
Seed | GBPUSD | AUDUSD | USDCHF | USDCAD | Interaction Score (= Δ(both vs max(htf,sess)))
```

**Interpretation:**
- Wenn USDCHF `interaction_score < -0.1` über ALLE 3 Seeds → **reproduzierbares Strukturproblem**
- Wenn USDCHF `interaction_score` schwankt zwischen Seeds → **Seed-Variance-Artefakt**
- Wenn andere Pairs ähnliches Pattern haben → **Universal-Multi-Filter-Problem**, nicht USDCHF-spezifisch

## Decision-Tree (NACH dem Run)

| Befund | Implikation für V1-Architektur |
|---|---|
| Reproduzierbar destructive bei USDCHF only | Pair-Sensitivity → ANN-017 V1-Architektur Per-Pair-Logic |
| Reproduzierbar destructive bei mehreren Pairs | Universal-Filter-Stack ist nicht robust → Filter-Mechanik überdenken |
| Nur Seed-Variance | Ensemble-of-Seeds oder Variance-Penalty im Training → V1.5+ |
| Gemischt | Beide Mechanismen wirken → genauere Tests in NB15c |

## Section 0 — Setup + Module Sync

In [ ]:
import os, sys, subprocess, gc, json, importlib
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    subprocess.run(['rm', '-rf', '/tmp/pace-algo'])
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ecoNC/pace-algo.git',
                    '/tmp/pace-algo'], check=True)
    subprocess.run(f'mkdir -p {PROJECT_ROOT}/core/eval {PROJECT_ROOT}/core/analysis {PROJECT_ROOT}/core/router',
                   shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/core/. {PROJECT_ROOT}/core/', shell=True)
    subprocess.run(f"find {PROJECT_ROOT}/core -type d -name __pycache__ -exec rm -rf {{}} +", shell=True)
    expected = [f'{PROJECT_ROOT}/core/analysis/diagnostic_decomposer.py']
    missing = [p for p in expected if not Path(p).exists()]
    if missing:
        raise SystemExit(f'SYNC FAILED: {missing}')
    print('Core synced.')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
importlib.invalidate_caches()

import numpy as np

SEEDS = [42, 1, 7]   # matched mit NB14f-v2 Multi-Seed-Set
RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'],
                                          text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb15b_{RUN_TIMESTAMP}_{GIT_COMMIT}'
print(f'EXPERIMENT_ID: {EXPERIMENT_ID}  |  seeds: {SEEDS}')

In [ ]:
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'lightgbm>=4.3', 'pyarrow>=15.0'], capture_output=True)

import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

from core import config as cfg
from core.config import (
    FX_TRAIN_SYMBOLS, FX_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END, HTF_CONTEXT_TIMEFRAMES,
)
from core.train.dataset import walk_forward_split, binary_label_for_long, NON_FEATURE_COLS
from core.analysis.probability_diagnostic import extract_premium_cluster
from core.analysis.diagnostic_decomposer import (
    filter_marginal_contributions, filter_interaction_score,
)

TF = '5m'
R_VALUE = 1.5
ALL_HOLDOUT = FX_HOLDOUT_SYMBOLS

OUTPUT_DIR = Path(PROJECT_ROOT) / 'results' / 'nb15b'
for sub in ('metrics', 'summaries', 'config_snapshots'):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
    'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
    'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
    'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
    'htf_ltf_agree_bull', 'htf_ltf_agree_bear', 'htf_ltf_counter_trend',
    'htf_ltf_alignment_score', 'ltf_rsi_minus_htf_rsi',
    'both_rsi_oversold', 'both_rsi_overbought', 'vol_pct_diff_htf',
    'both_high_vol', 'both_low_vol', 'pullback_in_bull', 'pullback_in_bear',
]
print(f'Hold-out pairs: {ALL_HOLDOUT}')
print(f'Output dir:     {OUTPUT_DIR}')

## Section 1 — Data Loading

In [ ]:
if IN_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        subprocess.run(['rsync', '-ah', f'{DRIVE_BACKUP_PROCESSED}/', f'{DATA_PROCESSED_LOCAL}/'])
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        subprocess.run(['rsync', '-ah', f'{EXT_DRIVE_BACKUP}/', f'{DATA_EXT}/'])
else:
    DATA_EXT = cfg.DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = cfg.DATA_PROCESSED

def load_ext(sym, tf=TF):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    if 'hit_bar_offset' not in df.columns:
        lp = DATA_PROCESSED_LOCAL / f'labels_{sym}_{tf}_R{R_VALUE}.parquet'
        if lp.exists():
            labels = pd.read_parquet(lp)
            if 'hit_bar_offset' in labels.columns:
                df = df.join(labels[['hit_bar_offset']], how='left')
        if 'hit_bar_offset' not in df.columns:
            df['hit_bar_offset'] = 24
        df['hit_bar_offset'] = df['hit_bar_offset'].fillna(24).astype('int32')
    return df

# Stack training pool (one-time)
frames = []
for s in FX_TRAIN_SYMBOLS:
    d = load_ext(s)
    if d is None or d.empty:
        raise SystemExit(f'Missing extended for {s}')
    float_cols = d.select_dtypes(include=['float64']).columns
    if len(float_cols) > 0:
        d = d.astype({c: 'float32' for c in float_cols}, copy=False)
    frames.append(d)
train_pool = pd.concat(frames, axis=0).sort_index().dropna(subset=FEATURE_COLS + ['label'])
del frames
gc.collect()

train_df, val_df, _ = walk_forward_split(train_pool, TRAIN_END, VAL_END)
del train_pool
gc.collect()

X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = binary_label_for_long(train_df['label']).values
X_val = val_df[FEATURE_COLS].values.astype(np.float32)
y_val = binary_label_for_long(val_df['label']).values
del train_df, val_df
gc.collect()

# Pre-load hold-out data (one-time)
holdout_data = {}
for sym in ALL_HOLDOUT:
    h = load_ext(sym)
    if h is None or h.empty:
        continue
    h = h.dropna(subset=FEATURE_COLS + ['label'])
    _, _, h_test = walk_forward_split(h, TRAIN_END, VAL_END)
    if h_test.empty:
        continue
    X_h = h_test[FEATURE_COLS].values.astype(np.float32)
    htf_align = (h_test['htf_1h_ema_alignment'].fillna(0).values > 0
                  if 'htf_1h_ema_alignment' in h_test.columns
                  else np.ones(len(h_test), dtype=bool))
    hours = np.asarray(h_test.index.hour)
    ny_mask = (hours >= 13) & (hours < 22)
    holdout_data[sym] = {
        'index':         h_test.index,
        'X':             X_h,
        'labels':        h_test['label'].values,
        'htf_mask':      htf_align,
        'ny_mask':       ny_mask,
    }
    del h
    gc.collect()

print(f'Train: {len(X_train):,} rows  VAL: {len(X_val):,} rows')
for sym, hd in holdout_data.items():
    print(f'  {sym}: {len(hd["X"]):,} test rows  '
          f'HTF-Confirm-share: {hd["htf_mask"].mean():.2%}  '
          f'NY-Session-share: {hd["ny_mask"].mean():.2%}')

## Section 2 — Multi-Seed Training Loop

Pro Seed: trainiere LightGBM, extrahiere Cluster, evaluiere alle 4 Pairs mit `filter_marginal_contributions`.

Pro Pair × Seed: berechne **Filter-Interaction-Score** = Δ(both vs max(htf, session)).

In [ ]:
BASE_PARAMS = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05, 'num_iterations': 30, 'lambda_l2': 0.5,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
    'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
    'deterministic': True,
}

all_marginals = []      # (seed, symbol, config, ...)
all_interactions = []   # (seed, symbol, base/htf/sess/both PFs, interaction_score, verdict)
seed_clusters = {}

for seed in SEEDS:
    print(f'\n=== Training seed={seed} ===')
    params = {**BASE_PARAMS, 'seed': seed}
    td = lgb.Dataset(X_train, label=y_train)
    vd = lgb.Dataset(X_val, label=y_val, reference=td)
    model = lgb.train(params, td, num_boost_round=30, valid_sets=[vd],
                       callbacks=[lgb.log_evaluation(period=0)])
    proba_val = model.predict(X_val)
    cluster_info = extract_premium_cluster(proba_val)
    cluster_cutoff = float(cluster_info['premium_cluster_value'])
    seed_clusters[seed] = {
        'cluster_value':  cluster_cutoff,
        'cluster_size':   cluster_info['premium_cluster_size'],
        'cluster_pct':    cluster_info['premium_cluster_pct'],
    }
    print(f'  Cluster: {cluster_cutoff:.4f} (size {cluster_info["premium_cluster_size"]}, '
          f'{cluster_info["premium_cluster_pct"]:.2f}%)')
    
    # Inferenz auf alle Hold-Out-Pairs
    for sym, hd in holdout_data.items():
        proba = model.predict(hd['X'])
        marg = filter_marginal_contributions(
            proba=proba,
            cluster_cutoff=cluster_cutoff,
            timestamps=hd['index'],
            labels_triple=hd['labels'],
            htf_confirm_mask=hd['htf_mask'],
            session_mask=hd['ny_mask'],
            R=R_VALUE,
        )
        marg['seed'] = seed
        marg['symbol'] = sym
        marg['cluster_cutoff'] = cluster_cutoff
        all_marginals.append(marg)
        
        # Filter-Interaction-Score
        fi = filter_interaction_score(marg)
        fi['seed'] = seed
        fi['symbol'] = sym
        fi['cluster_cutoff'] = cluster_cutoff
        all_interactions.append(fi)
        
        del proba
    del model, td, vd, proba_val
    gc.collect()
    print(f'  Done seed={seed}')

marg_full = pd.concat(all_marginals, ignore_index=True)
interaction_full = pd.DataFrame(all_interactions)
print(f'\nTotal rows: marg={len(marg_full)}  interaction={len(interaction_full)}')

## Section 3 — Interaction Score Matrix (Nico's Wunsch-Tabelle)

**Format:** `Seed × Symbol → Interaction-Score (= Δ(both vs max(htf, session)))`

In [ ]:
# Hauptmatrix: Interaction Score
interaction_matrix = interaction_full.pivot(index='seed', columns='symbol',
                                              values='interaction_score').round(3)
interaction_matrix = interaction_matrix[ALL_HOLDOUT]
print('=== Interaction Score Matrix (Seed × Symbol) ===')
print('Definition: Δ(both vs max(htf, session))')
print('   > +0.10 = additiv (Filter verstärken sich)')
print('  -0.10 to +0.10 = neutral')
print('   < -0.10 = destructive (Filter heben sich auf)')
display(interaction_matrix)

# Verdict-Matrix
verdict_matrix = interaction_full.pivot(index='seed', columns='symbol', values='verdict')
verdict_matrix = verdict_matrix[ALL_HOLDOUT]
print('\n=== Verdict Matrix (Seed × Symbol) ===')
display(verdict_matrix)

# Mean ± Std pro Symbol
stats_per_symbol = interaction_full.groupby('symbol').agg(
    n_seeds=('seed', 'count'),
    mean_interaction=('interaction_score', 'mean'),
    std_interaction=('interaction_score', 'std'),
    min_interaction=('interaction_score', 'min'),
    max_interaction=('interaction_score', 'max'),
).round(3)
stats_per_symbol = stats_per_symbol.reindex(ALL_HOLDOUT)
print('\n=== Per-Symbol Stability of Interaction Score ===')
display(stats_per_symbol)

## Section 4 — Full PF Matrices per Filter-Config

Pro Filter-Config × Symbol × Seed: PF. Damit sehen wir Stabilität/Instabilität pro Layer separat.

In [ ]:
for config in ['base', 'base_+htf', 'base_+session', 'base_+htf_+sess']:
    sub = marg_full[marg_full['config'] == config]
    pivot = sub.pivot(index='seed', columns='symbol', values='pf').round(3)
    pivot = pivot[ALL_HOLDOUT]
    print(f'\n=== PF — config: {config} ===')
    display(pivot)

## Section 5 — Seed-Variance pro Pair

Über alle 4 Filter-Configs: wie stabil ist PF pro Pair × Seed?

In [ ]:
variance_rows = []
for sym in ALL_HOLDOUT:
    for config in ['base', 'base_+htf', 'base_+session', 'base_+htf_+sess']:
        sub = marg_full[(marg_full['symbol'] == sym) & (marg_full['config'] == config)]
        if len(sub) == 0:
            continue
        pfs = sub['pf'].replace([np.inf, -np.inf], np.nan).dropna()
        if len(pfs) == 0:
            continue
        mean_pf = float(pfs.mean())
        std_pf  = float(pfs.std())
        cv = std_pf / mean_pf if mean_pf > 0 else float('inf')
        variance_rows.append({
            'symbol':  sym,
            'config':  config,
            'mean_pf': round(mean_pf, 3),
            'std_pf':  round(std_pf, 3),
            'cv_pf':   round(cv, 3),
            'n_seeds': len(pfs),
        })
variance_df = pd.DataFrame(variance_rows)

print('=== Seed-Variance pro Pair × Filter-Config ===')
for config in ['base', 'base_+htf', 'base_+session', 'base_+htf_+sess']:
    sub = variance_df[variance_df['config'] == config].set_index('symbol')
    print(f'\n--- {config} ---')
    display(sub[['mean_pf', 'std_pf', 'cv_pf']].reindex(ALL_HOLDOUT))

## Section 6 — Auto-Verdict (Architektur-Entscheidungs-Input)

In [ ]:
def auto_verdict(stats_per_symbol):
    """
    Klassifiziert was wir sehen. Vier mögliche Outcomes:
    
    1. 'destructive_reproducible_single_pair'
       — mean < -0.1 AND std < 0.1 AND nur 1 pair betroffen
       → Pair-Sensitivity, Per-Pair-Logic erwägen (mit ANN-016 Discipline)
    
    2. 'destructive_reproducible_multi_pair'
       — mean < -0.1 AND std < 0.1 auf >= 2 pairs
       → Universal-Filter-Stack ist nicht robust, Filter-Mechanik überdenken
    
    3. 'high_seed_variance'
       — std > 0.15 auf irgendeinem pair
       → Seed-Fragility, Lösung via Ensemble / Robustness-Selection / Variance-Penalty
    
    4. 'stable_universal'
       — alle pairs additive oder neutral, std klein
       → Filter-Stack funktioniert universal, USDCHF-Verdacht war falsch
    """
    destructive_pairs = stats_per_symbol[
        (stats_per_symbol['mean_interaction'] < -0.10) &
        (stats_per_symbol['std_interaction'].fillna(1.0) < 0.10)
    ]
    high_variance_pairs = stats_per_symbol[
        stats_per_symbol['std_interaction'].fillna(0.0) > 0.15
    ]
    
    if len(destructive_pairs) >= 2:
        verdict = 'destructive_reproducible_multi_pair'
        action = 'Universal Filter-Stack ist nicht robust. Filter-Mechanik in ANN-017 ueberdenken.'
    elif len(destructive_pairs) == 1:
        verdict = 'destructive_reproducible_single_pair'
        action = f'Pair-Sensitivity bei {list(destructive_pairs.index)}. ANN-017 Per-Pair-Decision mit ANN-016 Lock 5 Discipline.'
    elif len(high_variance_pairs) >= 1:
        verdict = 'high_seed_variance'
        action = f'Seed-Fragility bei {list(high_variance_pairs.index)}. Loesung: Ensemble-of-Seeds, Robustness-Selection, oder Variance-Penalty (Pin V1.5+ Architektur, NICHT per-pair-overrides).'
    else:
        verdict = 'stable_universal'
        action = 'Filter-Stack ist universal robust. USDCHF-Verdacht aus NB14f-v2 war Aggregation-Artefakt.'
    return verdict, action

verdict, action = auto_verdict(stats_per_symbol)
print('=' * 70)
print(f'AUTO-VERDICT: {verdict}')
print('=' * 70)
print(f'\nAction: {action}')
print('\nPer-Symbol stats:')
display(stats_per_symbol)

## Section 7 — Result Persistence + Auto-Push

In [ ]:
interaction_full.round(4).to_csv(OUTPUT_DIR / 'metrics' / f'interaction_per_seed_symbol_{RUN_DATE}.csv', index=False)
marg_full.round(4).to_csv(OUTPUT_DIR / 'metrics' / f'marginal_per_seed_symbol_{RUN_DATE}.csv', index=False)
interaction_matrix.to_csv(OUTPUT_DIR / 'metrics' / f'interaction_matrix_{RUN_DATE}.csv')
stats_per_symbol.to_csv(OUTPUT_DIR / 'summaries' / f'stats_per_symbol_{RUN_DATE}.csv')
variance_df.to_csv(OUTPUT_DIR / 'summaries' / f'seed_variance_{RUN_DATE}.csv', index=False)

def _to_json(o):
    if isinstance(o, dict):
        return {k: _to_json(v) for k, v in o.items()}
    if isinstance(o, pd.DataFrame):
        return o.reset_index().to_dict(orient='records')
    if isinstance(o, (np.floating, np.integer)):
        return float(o)
    return o

snapshot = {
    'experiment_id':       EXPERIMENT_ID,
    'run_date':            RUN_DATE,
    'git_commit':          GIT_COMMIT,
    'seeds':               SEEDS,
    'seed_clusters':       _to_json(seed_clusters),
    'interaction_matrix':  _to_json(interaction_matrix),
    'verdict_matrix':      _to_json(verdict_matrix),
    'stats_per_symbol':    _to_json(stats_per_symbol),
    'seed_variance':       _to_json(variance_df),
    'auto_verdict':        verdict,
    'action':              action,
    'interaction_full':    _to_json(interaction_full),
    'marginal_full':       _to_json(marg_full),
}
with open(OUTPUT_DIR / 'summaries' / f'nb15b_full_snapshot_{RUN_DATE}.json', 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)

with open(OUTPUT_DIR / 'config_snapshots' / f'{EXPERIMENT_ID}_config.json', 'w') as f:
    json.dump({
        'experiment_id': EXPERIMENT_ID,
        'seeds': SEEDS,
        'tf': TF,
        'fx_train_symbols': list(FX_TRAIN_SYMBOLS),
        'fx_holdout_symbols': list(FX_HOLDOUT_SYMBOLS),
    }, f, indent=2)

print(f'Results in {OUTPUT_DIR}')

In [ ]:
import shutil
if not IN_COLAB:
    print('Local — skip push.')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN') or userdata.get('GH_PAT')
    except Exception:
        GH_TOKEN = None
    if not GH_TOKEN:
        print('Kein GH-Token in Secrets.')
    else:
        TMP = Path('/tmp/pace-algo-push')
        if TMP.exists():
            shutil.rmtree(TMP)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@github.com/ecoNC/pace-algo.git', str(TMP)], check=True)
        subprocess.run(['git', '-C', str(TMP), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP), 'config', 'user.email', 'ecoNC@users.noreply.github.com'], check=True)
        copied = 0
        for f in sorted(OUTPUT_DIR.rglob(f'*{RUN_DATE}*')):
            if not f.is_file():
                continue
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied += 1
        for f in sorted((OUTPUT_DIR / 'config_snapshots').glob(f'*{EXPERIMENT_ID}*')):
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied += 1
        subprocess.run(['git', '-C', str(TMP), 'pull', '--rebase', '--quiet', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(TMP), 'add', 'results/'], check=True)
        st = subprocess.run(['git', '-C', str(TMP), 'status', '--porcelain'], capture_output=True, text=True)
        if not st.stdout.strip():
            print('Nothing to commit.')
        else:
            msg = f'NB15b Multi-Seed Interaction-Robustness {RUN_DATE} ({copied} files) — verdict: {verdict}'
            subprocess.run(['git', '-C', str(TMP), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP), 'rev-parse', '--short', 'HEAD'],
                                  capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP), 'push', 'origin', 'main'], check=True)
            print(f'Pushed as ecoNC ({sha})')
        shutil.rmtree(TMP)